# MVCAF EfficientNet Seed Sweep (GPU)

GPU 환경에서 `efficientnetv2_rw_s`만 고정하고 seed를 바꿔가며 `MVCAF_backbone_test_v1.0` 설정의 분산폭을 확인하는 노트북입니다.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from IPython.display import display

ROOT = Path.cwd().resolve().parent
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from models import EMAConfig, ModelEMA
from reproducibility import make_generator, seed_everything, seed_worker

DATA_DIR = ROOT / "data"
WEIGHT_DIR = ROOT / "outputs" / "weights"
EDA_DIR = ROOT / "outputs" / "eda_preprocessing"
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "IMG_SIZE": 224,
    "EPOCHS": 30,
    "EARLY_STOPPING_PATIENCE": 5,
    "LEARNING_RATE": 3e-4,
    "BATCH_SIZE": 16,
    "NUM_WORKERS": 8,
    "WEIGHT_DECAY": 1e-4,
    "MIXUP_ALPHA": 0.1,
    "MIXUP_PROB": 0.0,
    "MIN_LR": 1e-6,
    "EMA_DECAY": 0.999,
    "EMA_USE_FOR_EVAL": True,
    "TTA_CANDIDATES": [
        ["none"],
        ["none", "hflip"],
        ["none", "hflip", "crop95"],
    ],
    "VIDEO_AUG_ENABLE": True,
    "VIDEO_AUG_CACHE": True,
    "UNSTABLE_START_MIN_SEC": 0.5,
    "UNSTABLE_START_MAX_SEC": 1.0,
    "UNSTABLE_FRAMES_MIN": 2,
    "UNSTABLE_FRAMES_MAX": 3,
    "STABLE_END_MIN_SEC": 9.0,
    "STABLE_END_MAX_SEC": 10.0,
    "STABLE_FRAMES_PER_VIDEO": 2,
}

SEEDS = [42, 11, 77]
BACKBONE_NAME = "efficientnetv2_rw_s"
EXPERIMENT_NAME = "mv_caf_efficientnet_seed_sweep_gpu"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
assert device.type == "cuda", "GPU notebook입니다. CUDA 환경에서 실행해 주세요."


device: cuda


In [2]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {"stable": 0, "unstable": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row["id"])
        base_dir = self.root_dir
        if ("sample_dir" in self.df.columns) and pd.notna(row.get("sample_dir", np.nan)):
            base_dir = str(row["sample_dir"])
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return views
        return views, self.label_map[row["label"]]


def _extract_frame_by_sec(cap, sec, fps, frame_count):
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg, seed):
    keys = [
        "UNSTABLE_START_MIN_SEC",
        "UNSTABLE_START_MAX_SEC",
        "UNSTABLE_FRAMES_MIN",
        "UNSTABLE_FRAMES_MAX",
        "STABLE_END_MIN_SEC",
        "STABLE_END_MAX_SEC",
        "STABLE_FRAMES_PER_VIDEO",
    ]
    sig = {k: cfg.get(k) for k in keys}
    sig["SEED"] = seed
    return sig


def build_video_augmented_df(train_df, data_dir, cfg, seed):
    train_root = data_dir / "train"
    aug_root = data_dir / "train_video_aug"
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / f"aug_df_seed{seed}.csv"
    cache_meta = aug_root / f"cache_meta_seed{seed}.json"
    cache_sig = _video_aug_cache_signature(cfg, seed)

    if cfg.get("VIDEO_AUG_CACHE", True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get("signature") == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df["sample_dir"] = str(aug_root)
                    return cached_df
        except Exception as exc:
            print("video aug cache read failed:", exc)

    for p in aug_root.glob(f"AUGV_seed{seed}_*"):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(seed)
    stable_rows = []
    unstable_rows = []
    saved_idx = 0

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f"AUGV_seed{seed}_{saved_idx:07d}"
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / "front.png")
        Image.fromarray(img).save(out_dir / "top.png")
        row = {"id": aug_id, "label": label, "sample_dir": str(aug_root)}
        saved_idx += 1
        return row

    for sample_id in tqdm(train_df["id"].tolist(), desc=f"Video aug stable seed={seed}", dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / "simulation.mp4"
        if not video_path.exists():
            continue
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            continue
        img = _extract_last_frame(cap, frame_count)
        cap.release()
        if img is not None:
            stable_rows.append(save_aug(img, "stable"))

    unstable_ids = train_df.loc[train_df["label"] == "unstable", "id"].tolist()
    for sample_id in tqdm(unstable_ids, desc=f"Video aug unstable seed={seed}", dynamic_ncols=True, ascii=True):
        video_path = train_root / sample_id / "simulation.mp4"
        if not video_path.exists():
            continue
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps is None or fps <= 0 or frame_count <= 1:
            cap.release()
            continue
        duration = frame_count / fps
        low = cfg["UNSTABLE_START_MIN_SEC"]
        high = min(cfg["UNSTABLE_START_MAX_SEC"], max(0.0, duration - 1.0 / fps))
        if high <= low:
            cap.release()
            continue
        n_unstable = int(rng.integers(cfg["UNSTABLE_FRAMES_MIN"], cfg["UNSTABLE_FRAMES_MAX"] + 1))
        unstable_secs = rng.uniform(low, high, size=n_unstable)
        for sec in unstable_secs:
            img = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
            if img is not None:
                unstable_rows.append(save_aug(img, "unstable"))
        cap.release()

    stable_df = pd.DataFrame(stable_rows)
    unstable_df = pd.DataFrame(unstable_rows)
    if stable_df.empty or unstable_df.empty:
        return pd.DataFrame(columns=["id", "label", "sample_dir"])

    target_unstable = len(stable_df)
    if len(unstable_df) >= target_unstable:
        unstable_bal = unstable_df.sample(n=target_unstable, random_state=seed)
    else:
        unstable_bal = unstable_df.sample(n=target_unstable, replace=True, random_state=seed)

    aug_df = pd.concat([stable_df, unstable_bal], ignore_index=True)
    aug_df = aug_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    if cfg.get("VIDEO_AUG_CACHE", True):
        aug_df.to_csv(cache_csv, index=False)
        cache_meta.write_text(json.dumps({"signature": cache_sig}, ensure_ascii=False, indent=2))
    return aug_df


In [3]:
@dataclass(frozen=True)
class FlexibleCAFConfig:
    backbone_name: str = BACKBONE_NAME
    pretrained: bool = True
    attn_dim: int = 256
    num_heads: int = 8
    num_layers: int = 2
    dropout: float = 0.1
    classifier_hidden_dim: int = 512
    classifier_mid_dim: int = 128
    classifier_dropout: float = 0.3
    out_dim: int = 1


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens: torch.Tensor, kv_tokens: torch.Tensor) -> torch.Tensor:
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class MultiViewFlexibleCAF(nn.Module):
    def __init__(self, config: FlexibleCAFConfig):
        super().__init__()
        self.config = config
        self.backbone = timm.create_model(config.backbone_name, pretrained=config.pretrained, num_classes=0, global_pool="")
        self.feature_dim = getattr(self.backbone, "num_features")
        self.cnn_proj = nn.Conv2d(self.feature_dim, config.attn_dim, kernel_size=1, bias=False)
        self.token_proj = nn.Linear(self.feature_dim, config.attn_dim, bias=False)
        self.cross_12 = nn.ModuleList([CrossAttentionBlock(config.attn_dim, config.num_heads, config.dropout) for _ in range(config.num_layers)])
        self.cross_21 = nn.ModuleList([CrossAttentionBlock(config.attn_dim, config.num_heads, config.dropout) for _ in range(config.num_layers)])
        self.classifier = nn.Sequential(
            nn.Linear(config.attn_dim * 2, config.classifier_hidden_dim),
            nn.BatchNorm1d(config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.classifier_dropout),
            nn.Linear(config.classifier_hidden_dim, config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(config.classifier_mid_dim, config.out_dim),
        )

    def _to_tokens(self, feat: torch.Tensor) -> torch.Tensor:
        if feat.ndim == 4:
            if feat.shape[1] == self.feature_dim:
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
            if feat.shape[-1] == self.feature_dim:
                feat = feat.permute(0, 3, 1, 2)
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
        if feat.ndim == 3:
            if feat.shape[-1] == self.feature_dim:
                return self.token_proj(feat)
            if feat.shape[1] == self.feature_dim:
                return self.token_proj(feat.transpose(1, 2))
        raise ValueError(f"Unsupported feature shape: {tuple(feat.shape)}")

    def forward(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)
        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)
        feat = torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)
        return self.classifier(feat)


def mixup_multiview_batch(views, labels, alpha=0.2):
    if alpha <= 0:
        return views, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(labels.size(0), device=labels.device)
    mixed_views = [lam * v + (1.0 - lam) * v[index, :] for v in views]
    return mixed_views, labels, labels[index], lam


def train_one_epoch(model, loader, criterion, optimizer, device, mixup_alpha=0.2, mixup_prob=0.5, ema=None):
    model.train()
    train_loss = 0.0
    for views, labels in tqdm(loader, desc="Training", dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()
        optimizer.zero_grad()
        if mixup_alpha > 0 and np.random.rand() < mixup_prob:
            mixed_views, labels_a, labels_b, lam = mixup_multiview_batch(views, labels, alpha=mixup_alpha)
            outputs = model(mixed_views).view(-1)
            loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
        else:
            outputs = model(views).view(-1)
            loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if ema is not None:
            ema.update(model)
        train_loss += loss.item()
    return train_loss / max(len(loader), 1)


def validate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation", dynamic_ncols=True, ascii=True):
            views = [v.to(device) for v in views]
            logits = model(views).view(-1)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())
    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    p = np.clip(all_probs, 1e-15, 1 - 1e-15)
    logloss = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc = np.mean((all_probs > 0.5) == all_labels)
    return float(logloss), float(acc)


In [4]:
train_df_raw = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
val_df_raw = pd.read_csv(DATA_DIR / "dev.csv", encoding="utf-8-sig")
train_transform, test_transform = build_default_transforms(CFG["IMG_SIZE"])
print(len(train_df_raw), len(val_df_raw))

1000 100


In [5]:
def run_seed(seed: int):
    print(f"\n===== seed={seed} =====")
    seed_everything(seed)

    train_df_for_train = train_df_raw.copy()
    train_df_for_train["sample_dir"] = str(DATA_DIR / "train")
    if CFG["VIDEO_AUG_ENABLE"]:
        aug_df = build_video_augmented_df(train_df_raw, DATA_DIR, CFG, seed)
        if not aug_df.empty:
            train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

    val_df_for_eval = val_df_raw.copy()
    val_df_for_eval["sample_dir"] = str(DATA_DIR / "dev")

    train_dataset = MultiViewDataset(train_df_for_train, str(DATA_DIR / "train"), train_transform, is_test=False)
    val_dataset = MultiViewDataset(val_df_for_eval, str(DATA_DIR / "dev"), test_transform, is_test=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG["BATCH_SIZE"],
        shuffle=True,
        num_workers=CFG["NUM_WORKERS"],
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=make_generator(seed),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG["BATCH_SIZE"],
        shuffle=False,
        num_workers=CFG["NUM_WORKERS"],
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=make_generator(seed + 1),
    )

    model = MultiViewFlexibleCAF(FlexibleCAFConfig(backbone_name=BACKBONE_NAME)).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=CFG["LEARNING_RATE"], weight_decay=CFG["WEIGHT_DECAY"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["EPOCHS"], eta_min=CFG["MIN_LR"])
    ema = ModelEMA(model, EMAConfig(decay=CFG["EMA_DECAY"]))

    best_logloss = float("inf")
    best_acc = 0.0
    best_epoch = -1
    patience_left = CFG["EARLY_STOPPING_PATIENCE"]
    history = []
    ckpt_path = WEIGHT_DIR / f"{EXPERIMENT_NAME}_seed{seed}.pt"

    for epoch in range(1, CFG["EPOCHS"] + 1):
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            mixup_alpha=CFG["MIXUP_ALPHA"], mixup_prob=CFG["MIXUP_PROB"], ema=ema
        )
        eval_model = ema.ema_model if CFG["EMA_USE_FOR_EVAL"] else model
        val_logloss, val_acc = validate(eval_model, val_loader, device)
        improved = val_logloss < best_logloss
        if improved:
            best_logloss = val_logloss
            best_acc = val_acc
            best_epoch = epoch
            patience_left = CFG["EARLY_STOPPING_PATIENCE"]
            torch.save({
                "epoch": epoch,
                "seed": seed,
                "model_state_dict": model.state_dict(),
                "ema_state_dict": ema.ema_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "cfg": CFG,
                "backbone_name": BACKBONE_NAME,
                "val_logloss": val_logloss,
                "val_acc": val_acc,
            }, ckpt_path)
        else:
            patience_left -= 1

        scheduler.step()
        row = {
            "seed": seed,
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_logloss": float(val_logloss),
            "val_acc": float(val_acc),
            "improved": improved,
            "patience_left": patience_left,
        }
        history.append(row)
        print(row)

        if patience_left < 0:
            print(f"early stop seed={seed} epoch={epoch}")
            break

    hist_df = pd.DataFrame(history)
    hist_path = EDA_DIR / f"{EXPERIMENT_NAME}_seed{seed}_history.csv"
    hist_df.to_csv(hist_path, index=False)

    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_logloss": best_logloss,
        "best_val_acc": best_acc,
        "history_path": str(hist_path),
        "checkpoint_path": str(ckpt_path),
    }


In [6]:
results = []
for seed in SEEDS:
    results.append(run_seed(seed))

summary_df = pd.DataFrame(results).sort_values(["best_val_logloss", "best_val_acc"], ascending=[True, False]).reset_index(drop=True)
display(summary_df)
summary_path = EDA_DIR / f"{EXPERIMENT_NAME}_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("saved:", summary_path)


===== seed=42 =====


Validation: 100%|##########| 7/7 [00:00<00:00,  7.62it/s]


{'seed': 42, 'epoch': 1, 'train_loss': 0.29367763737335484, 'val_logloss': 0.6905449755259627, 'val_acc': 0.61, 'improved': True, 'patience_left': 5}


Validation: 100%|##########| 7/7 [00:00<00:00,  7.41it/s]


{'seed': 42, 'epoch': 2, 'train_loss': 0.14977747817186915, 'val_logloss': 0.6567504996898843, 'val_acc': 0.81, 'improved': True, 'patience_left': 5}


...

{'seed': 77, 'epoch': 18, 'train_loss': 0.02113745464912233, 'val_logloss': 0.27166288092553137, 'val_acc': 0.91, 'improved': False, 'patience_left': 0}


Validation: 100%|##########| 7/7 [00:00<00:00,  7.54it/s]


{'seed': 77, 'epoch': 19, 'train_loss': 0.014333366331656895, 'val_logloss': 0.32156037884187055, 'val_acc': 0.87, 'improved': False, 'patience_left': -1}
early stop seed=77 epoch=19


,seed,best_epoch,best_val_logloss,best_val_acc,history_path,checkpoint_path
0,77,13,0.128451,0.97,/media/hdd0/whyz/structure-stability/outputs/e...,/media/hdd0/whyz/structure-stability/outputs/w...
1,11,11,0.153991,0.94,/media/hdd0/whyz/structure-stability/outputs/e...,/media/hdd0/whyz/structure-stability/outputs/w...
2,42,12,0.156771,0.96,/media/hdd0/whyz/structure-stability/outputs/e...,/media/hdd0/whyz/structure-stability/outputs/w...


saved: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/mv_caf_efficientnet_seed_sweep_gpu_summary.csv


In [7]:
summary_df[["best_val_logloss", "best_val_acc"]].agg(["min", "max", "mean", "std"])

,best_val_logloss,best_val_acc
min,0.128451,0.940000
max,0.156771,0.970000
mean,0.146404,0.956667
std,0.015610,0.015275
